<a href="https://colab.research.google.com/github/nick-kann/Xatu-AI/blob/main/BuildDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
import sqlite3
import json
import pandas as pd
import requests
from IPython.display import clear_output

# **Creating the Dataset**:

The focus will be on games in the Gen 9 OU format since it is the most popular format that allows each player to see the opponent's entire team and choose their leading Pokemon. Only high elo games are going to be used in the data, as higher elo players typically use more logic when selecting their leading Pokemon. In contrast, lower elo players often choose the same Pokemon repeatedly or pick randomly, which complicates the model's learning process. The top 500 players in Gen 9 OU are typically > 1650 elo, but to get a little bit more data points, the elo cutoff is going to be at 1600 elo. The data will be obtained by making HTTP GET requests to the Pokemon Showdown server.

In [ ]:
base_url = "https://replay.pokemonshowdown.com/search.json?format=gen9ou"

all_data = []
last_uploadtime = None
total_fetched = 0

while True:
    if last_uploadtime is None:
        url = base_url
    else:
        url = f"{base_url}&before={last_uploadtime}"

    response = requests.get(url)

    if response.status_code != 200:
        print("Error fetching data:", response.status_code)
        break

    data = response.json()

    if not data:
        break

    all_data.extend(data)

    last_uploadtime = data[-1]["uploadtime"]

    total_fetched = len(all_data)
    print(f"\rTotal replays fetched: {total_fetched}", end='')

Total replays fetched: 677196

In [ ]:
df = pd.DataFrame(all_data)
df

,uploadtime,id,format,players,rating,private,password
0,1729038188,gen9ou-2223822396,[Gen 9] OU,"[tines, Sappkira]",1516.0,0,None
1,1729038086,gen9ou-2223822645,[Gen 9] OU,"[roba tussin, steve aokidogi]",1233.0,0,None
2,1729038065,gen9ou-2223820363,[Gen 9] OU,"[nuggasah, LaflaredaGod]",1777.0,0,None
3,1729038065,gen9ou-2223820284,[Gen 9] OU,"[Mky27, lokixenjoyer738]",1451.0,0,None
4,1729038041,gen9ou-2223822867,[Gen 9] OU,"[Calibold, Pffft is me]",1473.0,0,None
...,...,...,...,...,...,...,...
677191,1669316114,smogtours-gen9ou-662498,[Gen 9] OU,"[Charmflash, HarryBW247]",NaN,0,None
677192,1669315924,smogtours-gen9ou-662497,[Gen 9] OU,"[Charmflash, HarryBW247]",NaN,0,None
677193,1669313957,smogtours-gen9ou-662495,[Gen 9] OU,"[Vileman, BeatsBlack]",NaN,0,None
677194,1669313259,smogtours-gen9ou-662491,[Gen 9] OU,"[Vileman, BeatsBlack]",NaN,0,None


In [ ]:
# Dropping replays that have no associated elo rating
df = df.dropna(subset=['rating'])
df

,uploadtime,id,format,players,rating,private,password
0,1729038188,gen9ou-2223822396,[Gen 9] OU,"[tines, Sappkira]",1516.0,0,None
1,1729038086,gen9ou-2223822645,[Gen 9] OU,"[roba tussin, steve aokidogi]",1233.0,0,None
2,1729038065,gen9ou-2223820363,[Gen 9] OU,"[nuggasah, LaflaredaGod]",1777.0,0,None
3,1729038065,gen9ou-2223820284,[Gen 9] OU,"[Mky27, lokixenjoyer738]",1451.0,0,None
4,1729038041,gen9ou-2223822867,[Gen 9] OU,"[Calibold, Pffft is me]",1473.0,0,None
...,...,...,...,...,...,...,...
671345,1701532085,gen9ou-2003211714,[Gen 9] OU,"[mywifenkids, i am ass2]",1435.0,0,None
671346,1701532074,gen9ou-2003211704,[Gen 9] OU,"[ortegajd, Seltzer Time]",1359.0,0,None
671347,1701532061,gen9ou-2003211656,[Gen 9] OU,"[Ehdhdhdh, alle43]",1457.0,0,None
671348,1701532057,gen9ou-2003211428,[Gen 9] OU,"[Sknmdeelectricidad, Adel19]",1654.0,0,None


In [ ]:
# Filtering the dataframe to only contain games with >= 1600 elo
df_high_elo = df[df['rating'] >= 1600]
df_high_elo

,uploadtime,id,format,players,rating,private,password
2,1729038065,gen9ou-2223820363,[Gen 9] OU,"[nuggasah, LaflaredaGod]",1777.0,0,None
11,1729037647,gen9ou-2223818916,[Gen 9] OU,"[Johnny tots, nuggasah]",1737.0,0,None
12,1729037647,gen9ou-2223818488,[Gen 9] OU,"[LaflaredaGod, freakszn]",1795.0,0,None
16,1729037507,gen9ou-2223816039,[Gen 9] OU,"[LaflaredaGod, nuggasah]",1771.0,0,None
20,1729037229,gen9ou-2223813391,[Gen 9] OU,"[nuggasah, freakszn]",1791.0,0,None
...,...,...,...,...,...,...,...
671277,1701532579,gen9ou-2003212810,[Gen 9] OU,"[ruebs, Hoot-hoot Shiny]",1703.0,0,None
671284,1701532548,gen9ou-2003213133,[Gen 9] OU,"[alvar03, Kurosu eX]",1658.0,0,None
671301,1701532412,gen9ou-2003212100,[Gen 9] OU,"[TrepYT, StazMTA]",1621.0,0,None
671321,1701532248,gen9ou-2003211245,[Gen 9] OU,"[Msousagamer, repete64]",1678.0,0,None


In [ ]:
# Filtering the dataframe for low elo games as well just to test it out
df_low_elo = df[(df['rating'] >= 1150) & (df['rating'] <= 1300)]
df_low_elo

,uploadtime,id,format,players,rating,private,password
0,1729025646,gen9ou-2223696317,[Gen 9] OU,"[Talim1998, rosameltroso231]",1159.0,0,None
2,1729025645,gen9ou-2223705316,[Gen 9] OU,"[BravestSquirtle, dwaugh2]",1172.0,0,None
6,1729025505,gen9ou-2223703677,[Gen 9] OU,"[NOAHB2010, rostocks]",1161.0,0,None
10,1729025366,gen9ou-2223702413,[Gen 9] OU,"[4jne, Oowaza]",1245.0,0,None
12,1729025366,gen9ou-2223697098,[Gen 9] OU,"[arteries55, PulseCreature]",1237.0,0,None
...,...,...,...,...,...,...,...
670980,1701532131,gen9ou-2003212253,[Gen 9] OU,"[100kmperhourgranny, cvxbcxvb]",1224.0,0,None
670981,1701532113,gen9ou-2003211179,[Gen 9] OU,"[Pachydon, taeoop]",1203.0,0,None
670984,1701532105,gen9ou-2003211501,[Gen 9] OU,"[battleboy12, juiicyskiller]",1233.0,0,None
670986,1701532090,gen9ou-2003211115,[Gen 9] OU,"[Teddy123456789, random ttar fan]",1292.0,0,None


**With the high-elo replays collected, the next step is to obtain the specific game-data for each replay.**

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# Function to fetch a single game log
def fetch_game_log(game_id):
    url = f"https://replay.pokemonshowdown.com/{game_id}.json"
    response = requests.get(url)
    return response.json()

high_game_logs = []
high_ids = df_high_elo['id'].tolist()

n = len(high_ids)

# Use ThreadPoolExecutor to parallelize calls
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = []

    for id in high_ids:
        future = executor.submit(fetch_game_log, id)
        futures.append(future)

    for i, future in enumerate(as_completed(futures)):
        data = future.result()
        high_game_logs.append(data)
        print(f"\r{i + 1}/{n} games processed", end='')

print("\nAll games processed.")

81578/81578 games processed
All games processed.


**The 'high_game_logs' list will now be downloaded and saved so if the kernel crashes while processing, the process doesn't have to be completely restarted.**

In [ ]:
from google.colab import files

with open('high_elo_game_logs.json', 'w') as f:
    json.dump(high_game_logs, f)

files.download('high_elo_game_logs.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Function to fetch a single game log
# (with retry logic because it crashed many times before)
def fetch_game_log(game_id):
    url = f"https://replay.pokemonshowdown.com/{game_id}.json"

    session = requests.Session()
    retry = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)

    response = session.get(url, timeout=5)
    response.raise_for_status()
    return response.json()

low_game_logs = []
ids = df_low_elo['id'].tolist()

n = len(ids)

# Use ThreadPoolExecutor to parallelize calls
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = []

    for id in ids:
        future = executor.submit(fetch_game_log, id)
        futures.append(future)

    for i, future in enumerate(as_completed(futures)):
        data = future.result()
        low_game_logs.append(data)
        print(f"\r{i + 1}/{n} games processed", end='')

print("\nAll games processed.")

142717/142717 games processed
All games processed.


**'low_elo_logs' will also be downloaded just to be safe.**

In [ ]:
from google.colab import files

with open('low_elo_game_logs.json', 'w') as f:
    json.dump(low_game_logs, f)

files.download('low_elo_game_logs.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Run this next block only if continuing from downloaded lists**.

In [1]:
from google.colab import drive
import json
drive.mount('/content/drive')

# Edit with own file path if necessary
low_file_path = '/content/drive/My Drive/low_elo_game_logs.json'
high_file_path = '/content/drive/My Drive/high_elo_game_logs.json'

low_game_logs = json.load(open(low_file_path))
high_game_logs = json.load(open(high_file_path))

Mounted at /content/drive


In [3]:
print(f"High elo count: {len(high_game_logs)}")
print(f"Low elo count: {len(low_game_logs)}")

High elo count: 81578
Low elo count: 142717


**Now that all the games are processed, a function has to be written in order to extract each player's teams and leading Pokemon from the raw data.**

In [ ]:
import re

def extract_teams(battle_log: str):
    teams = {
        "p1": set(),
        "p2": set()
    }

    leading_pokemon = {
        "p1": None,
        "p2": None
    }

    # Pattern to find the full teams for both players
    poke_pattern = r'poke\|(p1|p2)\|([^|,]+)'
    poke_matches = re.findall(poke_pattern, battle_log)

    for player, pokemon in poke_matches:
        pokemon = pokemon.strip() # Remove newline characters
        # Zamazenta is named Zamazenta-* in game logs
        pokemon = re.sub(r'Zamazenta-\*', 'Zamazenta', pokemon)
        # Greninja is named Greninja-* in game logs
        pokemon = re.sub(r'Greninja-\*', 'Zamazenta', pokemon)
        if player == 'p1':
            teams["p1"].add(pokemon)
        elif player == 'p2':
            teams["p2"].add(pokemon)

    # Pattern to find the leading Pokemon first each player
    switch_pattern = r'switch\|(p1a|p2a): [^|]+\|([^|,]+)'
    switch_matches = re.findall(switch_pattern, battle_log)

    # Keep track of the count to get only the first two leading Pokémon
    count = 0
    for player, pokemon in switch_matches:
        pokemon = pokemon.strip()
        pokemon = re.sub(r'Zamazenta-\*', 'Zamazenta', pokemon)
        pokemon = re.sub(r'Greninja-\*', 'Zamazenta', pokemon)
        if count >= 2:
            break
        if player == 'p1a' and leading_pokemon["p1"] is None:
            leading_pokemon["p1"] = pokemon
            count += 1
        elif player == 'p2a' and leading_pokemon["p2"] is None:
            leading_pokemon["p2"] = pokemon
            count += 1

    return teams, leading_pokemon

**Another team extraction function will be written except with the ability to see player one and two's most common lead, based off their last 15 games.**

In [89]:
from collections import Counter
import re

no_log_players = set()
def convert_to_userid(username):
        lower_str = username.lower()
        cleaned_str = re.sub(r'[^a-z0-9]', '', lower_str)
        return cleaned_str

def extract_teams_extra(battle_log: str, p1_username, p2_username: str):

    p1_username = convert_to_userid(p1_username)
    p2_username = convert_to_userid(p2_username)

    if p1_username in no_log_players or p2_username in no_log_players:
        raise Exception("No log player found")

    teams = {
        "p1_team": set(),
        "p2_team": set()
    }

    leading_pokemon = {
        "p1_lead": None,
        "p2_lead": None
    }

    common_leads = {
        "p1_common": None,
        "p2_common": None
    }

    url = f'https://replay.pokemonshowdown.com/search.json?user={p1_username}&format=gen9ou'
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Failed to retrieve game logs for {p1_username}")
        return "none"
    game_logs = response.json()
    p1_last_7_game_ids = [game['id'] for game in game_logs[:7]]

    # Find the most common lead of player 1 in their last 15 games
    p1_leads = []
    for game_id in p1_last_7_game_ids:

        game_url = f'https://replay.pokemonshowdown.com/{game_id}.log'
        game_response = requests.get(game_url)
        if game_response.status_code == 200:
            battle_log = game_response.text
            switch_pattern = r'switch\|(p1a): [^|]+\|([^|,]+)'
            switch_matches = re.findall(switch_pattern, battle_log)
            if switch_matches:
                lead_pokemon = switch_matches[0][1].strip()
                lead_pokemon = re.sub(r'Zamazenta-\*', 'Zamazenta', lead_pokemon)
                p1_leads.append(lead_pokemon)
    if len(p1_leads) == 0:
        no_log_players.add(p1_username)
        raise Exception(f"No saved past game logs for p1: {p1_username}")
    common_leads["p1_common"] = Counter(p1_leads).most_common(1)[0][0]

    url = f'https://replay.pokemonshowdown.com/search.json?user={p2_username}&format=gen9ou'
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Failed to retrieve game logs for {p2_username}")
        return "none"
    game_logs = response.json()
    p2_last_7_game_ids = [game['id'] for game in game_logs[:7]]

    # Find the most common lead of player 2 in their last 15 games
    p2_leads = []
    for game_id in p2_last_7_game_ids:
        game_url = f'https://replay.pokemonshowdown.com/{game_id}.log'
        game_response = requests.get(game_url)
        if game_response.status_code == 200:
            battle_log = game_response.text
            switch_pattern = r'switch\|(p2a): [^|]+\|([^|,]+)'
            switch_matches = re.findall(switch_pattern, battle_log)
            if switch_matches:
                lead_pokemon = switch_matches[0][1].strip()
                lead_pokemon = re.sub(r'Zamazenta-\*', 'Zamazenta', lead_pokemon)
                p2_leads.append(lead_pokemon)
    if len(p2_leads) == 0:
        no_log_players.add(p2_username)
        raise Exception(f"No saved past game logs for p2: {p2_username}")
    common_leads["p2_common"] = Counter(p2_leads).most_common(1)[0][0]

    # Pattern to find the full teams for both players
    poke_pattern = r'poke\|(p1|p2)\|([^|,]+)'
    poke_matches = re.findall(poke_pattern, battle_log)

    for player, pokemon in poke_matches:
        pokemon = pokemon.strip() # Remove newline characters
        # Zamazenta is named Zamazenta-* in game logs
        pokemon = re.sub(r'Zamazenta-\*', 'Zamazenta', pokemon)
        # Greninja is named Greninja-* in game logs
        pokemon = re.sub(r'Greninja-\*', 'Zamazenta', pokemon)
        if player == 'p1':
            teams["p1_team"].add(pokemon)
        elif player == 'p2':
            teams["p2_team"].add(pokemon)

    # Pattern to find the leading Pokemon first each player
    switch_pattern = r'switch\|(p1a|p2a): [^|]+\|([^|,]+)'
    switch_matches = re.findall(switch_pattern, battle_log)

    # Keep track of the count to get only the first two leading Pokémon
    count = 0
    for player, pokemon in switch_matches:
        pokemon = pokemon.strip()
        pokemon = re.sub(r'Zamazenta-\*', 'Zamazenta', pokemon)
        pokemon = re.sub(r'Greninja-\*', 'Zamazenta', pokemon)
        if count >= 2:
            break
        if player == 'p1a' and leading_pokemon["p1_lead"] is None:
            leading_pokemon["p1_lead"] = pokemon
            count += 1
        elif player == 'p2a' and leading_pokemon["p2_lead"] is None:
            leading_pokemon["p2_lead"] = pokemon
            count += 1

    return teams, leading_pokemon, common_leads

In [105]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_games_multithread(game_logs, max_workers=1):
    total_games = len(game_logs)
    results = []
    errors = []

    # Use ThreadPoolExecutor to parallelize extraction
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []

        for i, game in enumerate(game_logs):
            future = executor.submit(extract_teams_extra, game['log'],
                                     game['players'][0], game['players'][1])
            futures.append(future)

        for i, future in enumerate(as_completed(futures)):
            try:
                result = future.result()
                results.append(result)
            except Exception as e:
                errors.append(f"\nError processing game {i + 1}: {e}")
            print(f"\r{i + 1}/{total_games} games processed", end='')

    print("\nAll games processed.")
    return results, errors

In [106]:
high_game_teams, errors = process_games_multithread(high_game_logs[54440:54449])

9/9 games processed
All games processed.


In [108]:
len(high_game_teams)

5

In [ ]:
spec_game = high_game_logs[54440:54449]
try:
    extract_teams_extra(spec_game[0]['log'], spec_game[0]['players'][0], spec_game[0]['players'][1])
except Exception as e:
    print(e)
    print("whoops")

gen9ou-2110462729
yep
nah_here
flow%20like%20a%20river
['gen9ou-2186845445', 'gen9ou-2186618536', 'gen9ou-2186616659', 'gen9ou-2186611857', 'gen9ou-2186610524', 'gen9ou-2186606722', 'gen9ou-2186603555']
after here
[]
list index out of range
whoops


In [ ]:
high_game_teams = process_games_multithread(high_game_logs[54440::])

here
here
here
here
here
here
here
here
here
here


KeyboardInterrupt: 

In [ ]:
high_game_teams = []

for idx, game in enumerate(high_game_logs):
    high_game_teams.append(extract_teams_extra(game['log'], game['players'][0], game['players'][1]))
    print(f"\rProcessing game {idx + 1}/{len(high_game_logs)}", end="")
print("\nAll games processed.")

Processing game 4/81578

KeyboardInterrupt: 

In [ ]:
low_game_teams = [extract_teams(game['log']) for game in high_game_logs]

In [ ]:
len(high_game_logs)

81339

In [ ]:
import csv

with open('/content/dataset.csv', mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(["id", "p1_poke1", "p1_poke2", "p1_poke3", "p1_poke4",
                     "p1_poke5", "p1_poke6", "p2_poke1", "p2_poke2", "p2_poke3",
                     "p2_poke4", "p2_poke5", "p2_poke6", "p1_choice", "p2_choice"])
    id = 1
    for teams, choices in game_teams:
        row = []
        row.append(id)
        id += 1
        for team in teams:
            for poke in teams[team]:
                row.append(poke)
        for choice in choices:
            row.append(choices[choice])
        writer.writerow(row)

In [ ]:
files.download('/content/dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>